In [ ]:
"""
============================================================
  CNN vs. Regular Neural Network — Classroom Demo
  Dataset: Fashion-MNIST (10 classes, 70,000 28×28 grayscale images)
============================================================

What this script shows
-----------------------
1. A plain "Dense / Fully-Connected" Neural Network (NN)
   • Every pixel is flattened into a 1-D vector.
   • The network has no idea that neighbouring pixels are related.

2. A Convolutional Neural Network (CNN)
   • Uses Conv2D + MaxPooling layers to learn *local* spatial patterns.
   • Far fewer trainable parameters per layer, yet much better accuracy.

Both models are trained for the same number of epochs so the
comparison is fair.  After training, a summary table and two
side-by-side accuracy/loss plots are displayed.
"""

import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ──────────────────────────────────────────────
#  0.  Reproducibility
# ──────────────────────────────────────────────
tf.random.set_seed(42)
np.random.seed(42)

# ──────────────────────────────────────────────
#  1.  Load & pre-process Fashion-MNIST
# ──────────────────────────────────────────────
print("\n📦  Loading Fashion-MNIST dataset …")
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

CLASS_NAMES = [
    "T-shirt", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal",  "Shirt",   "Sneaker",  "Bag",   "Boot"
]
NUM_CLASSES = 10
EPOCHS      = 10        # increase for better accuracy; keep low for a quick demo
BATCH_SIZE  = 128

# Normalise pixel values to [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32")  / 255.0

# Fashion-MNIST is (N, 28, 28) — add channel dim → (N, 28, 28, 1) for Conv2D
x_train = x_train[..., np.newaxis]
x_test  = x_test[...,  np.newaxis]

# Labels → one-hot
y_train_oh = keras.utils.to_categorical(y_train, NUM_CLASSES)
y_test_oh  = keras.utils.to_categorical(y_test,  NUM_CLASSES)

print(f"   Training samples : {len(x_train):,}")
print(f"   Test samples     : {len(x_test):,}")
print(f"   Image shape      : {x_train.shape[1:]}")


In [ ]:

# ──────────────────────────────────────────────
#  2.  Dataset preview
# ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle("Fashion-MNIST — Sample Images", fontsize=14, fontweight="bold")
indices = np.random.choice(len(x_train), 16, replace=False)
for ax, idx in zip(axes.flat, indices):
    ax.imshow(x_train[idx].squeeze(), cmap="gray")
    ax.set_title(CLASS_NAMES[int(y_train[idx])], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.savefig("fashion_mnist_samples.png", dpi=120)
plt.show()
print("\n🖼   Sample grid saved → fashion_mnist_samples.png")


In [ ]:

# ──────────────────────────────────────────────
#  3.  Model A — Regular (Dense) Neural Network
# ──────────────────────────────────────────────
print("\n" + "="*60)
print("  MODEL A  —  Regular (Fully-Connected) Neural Network")
print("="*60)

def build_regular_nn(input_shape, num_classes):
    """
    A standard multi-layer perceptron.
    The input image is FLATTENED first — all spatial structure is lost.
    """
    model = keras.Sequential([
        # ── Input ──────────────────────────────────────────────────
        layers.Input(shape=input_shape),

        # Flatten 32×32×3 → 3072 values (spatial info discarded!)
        layers.Flatten(),

        # ── Hidden layers ──────────────────────────────────────────
        layers.Dense(512, activation="relu"),
        layers.Dropout(0.3),

        layers.Dense(256, activation="relu"),
        layers.Dropout(0.3),

        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),

        # ── Output ─────────────────────────────────────────────────
        layers.Dense(num_classes, activation="softmax"),
    ], name="Regular_NN")
    return model


In [ ]:

nn_model = build_regular_nn(x_train.shape[1:], NUM_CLASSES)
nn_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
nn_model.summary()

print("\n⏱   Training Regular NN …")
t0 = time.time()
nn_history = nn_model.fit(
    x_train, y_train_oh,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    verbose=1,
)
nn_train_time = time.time() - t0

nn_loss, nn_acc = nn_model.evaluate(x_test, y_test_oh, verbose=0)
print(f"\n✅  Regular NN  — Test accuracy: {nn_acc:.4f}  |  Test loss: {nn_loss:.4f}")


In [ ]:

# ──────────────────────────────────────────────
#  4.  Model B — Convolutional Neural Network
# ──────────────────────────────────────────────
print("\n" + "="*60)
print("  MODEL B  —  Convolutional Neural Network (CNN)")
print("="*60)

def build_cnn(input_shape, num_classes):
    """
    A CNN uses small filters (kernels) that slide across the image,
    detecting edges, textures, and shapes — preserving spatial context.
    Max-pooling then downsamples the feature maps progressively.
    """
    model = keras.Sequential([
        # ── Input ──────────────────────────────────────────────────
        layers.Input(shape=input_shape),

        # ── Convolutional Block 1 ───────────────────────────────────
        # 32 filters, each 3×3  — learns low-level features (edges)
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),   # 32×32 → 16×16
        layers.Dropout(0.25),

        # ── Convolutional Block 2 ───────────────────────────────────
        # 64 filters — learns mid-level patterns (curves, corners)
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),   # 16×16 → 8×8
        layers.Dropout(0.25),

        # ── Convolutional Block 3 ───────────────────────────────────
        # 128 filters — learns high-level features (object parts)
        layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),   # 8×8 → 4×4
        layers.Dropout(0.25),

        # ── Classifier Head ─────────────────────────────────────────
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax"),
    ], name="CNN")
    return model

cnn_model = build_cnn(x_train.shape[1:], NUM_CLASSES)
cnn_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
cnn_model.summary()

print("\n⏱   Training CNN …")
t0 = time.time()
cnn_history = cnn_model.fit(
    x_train, y_train_oh,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    verbose=1,
)
cnn_train_time = time.time() - t0

cnn_loss, cnn_acc = cnn_model.evaluate(x_test, y_test_oh, verbose=0)
print(f"\n✅  CNN  — Test accuracy: {cnn_acc:.4f}  |  Test loss: {cnn_loss:.4f}")



In [ ]:
# ──────────────────────────────────────────────
#  5.  Summary table
# ──────────────────────────────────────────────
nn_params  = nn_model.count_params()
cnn_params = cnn_model.count_params()

print("\n" + "="*60)
print("  COMPARISON SUMMARY")
print("="*60)
print(f"{'Metric':<30} {'Regular NN':>12} {'CNN':>12}")
print("-"*56)
print(f"{'Trainable parameters':<30} {nn_params:>12,} {cnn_params:>12,}")
print(f"{'Training time (s)':<30} {nn_train_time:>12.1f} {cnn_train_time:>12.1f}")
print(f"{'Test accuracy':<30} {nn_acc:>12.4f} {cnn_acc:>12.4f}")
print(f"{'Test loss':<30} {nn_loss:>12.4f} {cnn_loss:>12.4f}")
print("="*60)



In [ ]:
# ──────────────────────────────────────────────
#  6.  Training curves
# ──────────────────────────────────────────────
fig = plt.figure(figsize=(15, 10))
fig.suptitle("CNN vs. Regular Neural Network — Training Comparison\n(CIFAR-10)",
             fontsize=15, fontweight="bold", y=1.01)

gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

# Accuracy — train
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(nn_history.history["accuracy"],  label="NN  train", color="#E07B54", lw=2)
ax1.plot(cnn_history.history["accuracy"], label="CNN train", color="#4C72B0", lw=2)
ax1.set_title("Training Accuracy")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Accuracy")
ax1.legend(); ax1.grid(alpha=0.3)

# Accuracy — validation
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(nn_history.history["val_accuracy"],  label="NN  val", color="#E07B54", lw=2, linestyle="--")
ax2.plot(cnn_history.history["val_accuracy"], label="CNN val", color="#4C72B0", lw=2, linestyle="--")
ax2.set_title("Validation Accuracy")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
ax2.legend(); ax2.grid(alpha=0.3)

# Loss — train
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(nn_history.history["loss"],  label="NN  train", color="#E07B54", lw=2)
ax3.plot(cnn_history.history["loss"], label="CNN train", color="#4C72B0", lw=2)
ax3.set_title("Training Loss")
ax3.set_xlabel("Epoch"); ax3.set_ylabel("Loss")
ax3.legend(); ax3.grid(alpha=0.3)

# Loss — validation
ax4 = fig.add_subplot(gs[1, 1])
ax4.plot(nn_history.history["val_loss"],  label="NN  val", color="#E07B54", lw=2, linestyle="--")
ax4.plot(cnn_history.history["val_loss"], label="CNN val", color="#4C72B0", lw=2, linestyle="--")
ax4.set_title("Validation Loss")
ax4.set_xlabel("Epoch"); ax4.set_ylabel("Loss")
ax4.legend(); ax4.grid(alpha=0.3)

plt.savefig("training_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print("\n📊  Training curves saved → training_curves.png")



In [ ]:
# ──────────────────────────────────────────────
#  7.  Per-class prediction comparison
# ──────────────────────────────────────────────
nn_preds  = np.argmax(nn_model.predict(x_test,  verbose=0), axis=1)
cnn_preds = np.argmax(cnn_model.predict(x_test, verbose=0), axis=1)
true_labels = y_test.flatten()

nn_per_class  = [np.mean(nn_preds[true_labels == c]  == c) for c in range(NUM_CLASSES)]
cnn_per_class = [np.mean(cnn_preds[true_labels == c] == c) for c in range(NUM_CLASSES)]

x = np.arange(NUM_CLASSES)
width = 0.38

fig, ax = plt.subplots(figsize=(13, 5))
bars1 = ax.bar(x - width/2, nn_per_class,  width, label="Regular NN", color="#E07B54", alpha=0.85)
bars2 = ax.bar(x + width/2, cnn_per_class, width, label="CNN",        color="#4C72B0", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right")
ax.set_ylabel("Accuracy")
ax.set_title("Per-Class Test Accuracy — Regular NN vs. CNN", fontsize=13, fontweight="bold")
ax.legend(); ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, 1)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=7)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=7)

plt.tight_layout()
plt.savefig("per_class_accuracy.png", dpi=120)
plt.show()
print("📊  Per-class accuracy chart saved → per_class_accuracy.png")



In [ ]:
# ──────────────────────────────────────────────
#  8.  Show random predictions side-by-side
# ──────────────────────────────────────────────
sample_idx = np.random.choice(len(x_test), 12, replace=False)
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
fig.suptitle("Random Predictions — same image, different model predictions (green=correct, red=wrong)", fontsize=12, fontweight="bold")

for col, idx in enumerate(sample_idx):
    true     = CLASS_NAMES[true_labels[idx]]
    nn_pred  = CLASS_NAMES[nn_preds[idx]]
    cnn_pred = CLASS_NAMES[cnn_preds[idx]]

    # col 0-5 → top row (NN) and bottom row (CNN) use the same column index
    # sample_idx has 12 items; map items 0-5 to cols 0-5, items 6-11 also to cols 0-5
    c = col % 6   # actual axes column (always 0-5)
    r = col // 6  # which pair-row: 0 = NN top / CNN bottom, 1 = second pair (not used here)

    ax_top = axes[0, c]
    ax_bot = axes[1, c]

    ax_top.imshow(x_test[idx].squeeze(), cmap="gray")
    top_colour = "green" if nn_pred == true else "red"
    ax_top.set_title(f"NN: {nn_pred}", color=top_colour, fontsize=8, fontweight="bold")
    ax_top.set_xlabel(f"True: {true}", fontsize=7)
    ax_top.axis("off")

    ax_bot.imshow(x_test[idx].squeeze(), cmap="gray")
    bot_colour = "green" if cnn_pred == true else "red"
    ax_bot.set_title(f"CNN: {cnn_pred}", color=bot_colour, fontsize=8, fontweight="bold")
    ax_bot.axis("off")

    if col == 5:   # only iterate the first 6 samples (one per column)
        break

axes[0, 0].set_ylabel("Regular NN", fontsize=11, fontweight="bold", color="#E07B54", labelpad=8)
axes[1, 0].set_ylabel("CNN",        fontsize=11, fontweight="bold", color="#1D9E75", labelpad=8)

plt.tight_layout()
plt.savefig("sample_predictions.png", dpi=120)
plt.show()
print("📊  Sample predictions saved → sample_predictions.png")

print("\n🎓  Demo complete!  Key takeaways:")
print("   • The CNN preserves spatial structure → higher accuracy.")
print("   • The Regular NN treats every pixel independently → misses patterns.")
print("   • CNNs often use *fewer* parameters yet achieve *better* results.")
print("   • Both models improve with more epochs and data augmentation.\n")